In [4]:
# random_forest.py
from pathlib import Path
import math
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.ensemble import RandomForestClassifier

PROJECT = Path.cwd().parents[0]
PROCESSED = PROJECT / 'data' / 'processed'
OUTPUT_CSV = PROJECT / "results" / "data"
OUTPUT_CSV.mkdir(parents=True, exist_ok=True)

train_csv = pd.read_csv(PROCESSED / 'train.csv')
valid_csv = pd.read_csv(PROCESSED / 'valid.csv')
test_csv  = pd.read_csv(PROCESSED / 'test.csv')

target_col = "SOURCE"
print("🎯 Using target:", target_col)

def encode_target(series: pd.Series):
    series = series.astype(str)
    classes = sorted(series.unique())
    mapping = {cls: i for i, cls in enumerate(classes)}
    print("🔑 Target mapping:", mapping)
    return series.map(mapping).astype(int), mapping

y_train, target_mapping = encode_target(train_csv[target_col])
y_valid = valid_csv[target_col].astype(str).map(target_mapping).astype(int)
y_test  = test_csv[target_col].astype(str).map(target_mapping).astype(int)

X_train = train_csv.drop(columns=[target_col])
X_valid = valid_csv.drop(columns=[target_col])
X_test  = test_csv.drop(columns=[target_col])

cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
for c in cat_cols:
    X_train[c] = X_train[c].astype(str)
    X_valid[c] = X_valid[c].astype(str)
    X_test[c]  = X_test[c].astype(str)

X_train_enc = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_valid_enc = pd.get_dummies(X_valid, columns=cat_cols, drop_first=True)
X_test_enc  = pd.get_dummies(X_test,  columns=cat_cols, drop_first=True)

X_valid_enc = X_valid_enc.reindex(columns=X_train_enc.columns, fill_value=0)
X_test_enc  = X_test_enc.reindex(columns=X_train_enc.columns, fill_value=0)

print("Encoded X shapes:", X_train_enc.shape, X_valid_enc.shape, X_test_enc.shape)

def compute_metrics(y_true, y_pred, model_name, split_name):
    acc = (y_pred == y_true).mean()

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 1) & (y_pred == 0)).sum())
    fn = int(((y_true == 0) & (y_pred == 1)).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score  = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    print(f"\n=== {model_name} – {split_name} ===")
    print(f"{split_name} accuracy : {acc:.4f}")
    print(f"{split_name} precision: {precision:.4f}")
    print(f"{split_name} recall   : {recall:.4f}")
    print(f"{split_name} f1-score : {f1_score:.4f}")
    print(f"{split_name} confusion matrix (tn, fp, fn, tp):")
    print(f"  TN={tn}, FP={fp}, FN={fn}, TP={tp}")

    return {
        "model": model_name,
        "split": split_name,
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1_score),
    }

# ========= TUNING RF =========
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 5, 10],
    "max_features": ["sqrt", "log2"],
}

best_acc = -1
best_params = None
best_model = None

print("\n🔎 Tuning RandomForest trên tập Valid...")

for n_est in param_grid["n_estimators"]:
    for max_depth in param_grid["max_depth"]:
        for max_feat in param_grid["max_features"]:
            rf = RandomForestClassifier(
                n_estimators=n_est,
                max_depth=max_depth,
                max_features=max_feat,
                random_state=42,
                n_jobs=-1,
            )
            rf.fit(X_train_enc, y_train)
            y_valid_pred = rf.predict(X_valid_enc)
            acc = (y_valid_pred == y_valid).mean()
            print(f"  n_estimators={n_est}, max_depth={max_depth}, max_features={max_feat} -> valid acc={acc:.4f}")

            if acc > best_acc:
                best_acc = acc
                best_params = (n_est, max_depth, max_feat)
                best_model = rf

print("\n✅ Best RandomForest trên Valid:")
print(f"  n_estimators={best_params[0]}, max_depth={best_params[1]}, max_features={best_params[2]}, valid acc={best_acc:.4f}")

# ========= ĐÁNH GIÁ & LƯU =========
all_scores = []

# 1) Train
y_train_pred = best_model.predict(X_train_enc)
metrics_train = compute_metrics(
    y_train, y_train_pred,
    model_name="random_forest",
    split_name="Train"
)
all_scores.append(metrics_train)

# 2) Valid
y_valid_pred = best_model.predict(X_valid_enc)
metrics_valid = compute_metrics(
    y_valid, y_valid_pred,
    model_name="random_forest",
    split_name="Valid"
)
all_scores.append(metrics_valid)

# 3) Test
y_test_pred = best_model.predict(X_test_enc)
metrics_test = compute_metrics(
    y_test, y_test_pred,
    model_name="random_forest",
    split_name="Test"
)
all_scores.append(metrics_test)

scores_df = pd.DataFrame(all_scores)
scores_file = OUTPUT_CSV / "model_scores.csv"
if scores_file.exists():
    scores_df.to_csv(scores_file, mode="a", header=False, index=False, encoding="utf-8-sig")
else:
    scores_df.to_csv(scores_file, index=False, encoding="utf-8-sig")

print("\n✅ Đã ghi thêm metric RandomForest vào:", scores_file)


🎯 Using target: SOURCE
🔑 Target mapping: {'in': 0, 'out': 1}
Encoded X shapes: (3088, 10) (662, 10) (662, 10)

🔎 Tuning RandomForest trên tập Valid...
  n_estimators=100, max_depth=None, max_features=sqrt -> valid acc=0.7628
  n_estimators=100, max_depth=None, max_features=log2 -> valid acc=0.7628
  n_estimators=100, max_depth=5, max_features=sqrt -> valid acc=0.7296
  n_estimators=100, max_depth=5, max_features=log2 -> valid acc=0.7296
  n_estimators=100, max_depth=10, max_features=sqrt -> valid acc=0.7568
  n_estimators=100, max_depth=10, max_features=log2 -> valid acc=0.7568
  n_estimators=200, max_depth=None, max_features=sqrt -> valid acc=0.7704
  n_estimators=200, max_depth=None, max_features=log2 -> valid acc=0.7704
  n_estimators=200, max_depth=5, max_features=sqrt -> valid acc=0.7266
  n_estimators=200, max_depth=5, max_features=log2 -> valid acc=0.7266
  n_estimators=200, max_depth=10, max_features=sqrt -> valid acc=0.7508
  n_estimators=200, max_depth=10, max_features=log2 -